# Traffic Volume Forecasting

## Project Overview

Forecasts **daily traffic volume** (vehicles) over a 14-day horizon. Synthetic data spans 2 years with strong weekly cycles, seasonal variation, and holiday effects.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily traffic counts, predict the next 14 days. Transportation agencies need traffic forecasts for congestion management, infrastructure planning, and signal timing optimization.

## Why This Project Matters

Traffic congestion costs billions in lost productivity and fuel. Accurate volume forecasts enable proactive traffic management, construction scheduling, and public transit planning.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily traffic volume
- Strong weekly cycle (high weekdays, low weekends)
- Moderate seasonal variation (summer travel)
- Holiday dips (commuter traffic drops)
- Slight upward trend (urbanization)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `traffic_vol` | Daily traffic volume (vehicle count) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'traffic_vol'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 50000 + 5 * t  # urbanization growth
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 8000
    elif dow == 5: weekly[i] = -3000
    else: weekly[i] = -8000

# Summer travel increase
seasonal = 3000 * np.sin(2 * np.pi * (t - 100) / 365)

# Holiday dips (commuter absence)
holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if (m == 12 and d >= 24) or (m == 1 and d <= 2): holiday[i] = -15000
    elif m == 11 and 22 <= d <= 28: holiday[i] = -10000
    elif m == 7 and 3 <= d <= 5: holiday[i] = -8000

noise = np.random.normal(0, 2000, N_POINTS)
traffic_vol = base + weekly + seasonal + holiday + noise
traffic_vol = np.maximum(traffic_vol, 10000).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'traffic_vol': traffic_vol})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,traffic_vol
0,2022-01-01,30027
1,2022-01-02,23755
2,2022-01-03,56326
3,2022-01-04,58076
4,2022-01-05,54562


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('traffic_vol Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("traffic_vol Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

traffic_vol Statistics:
count      730.000000
mean     55242.380822
std       7744.800607
min      23755.000000
25%      49830.500000
50%      57669.500000
75%      60900.500000
max      69612.000000
Name: traffic_vol, dtype: float64

CV: 0.140
Skewness: -1.016


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:     9086.9 | RMSE:    11632.1 | MAPE: 23.33%
Seasonal Naive (7)             | MAE:     8287.5 | RMSE:    10911.4 | MAPE: 21.97%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared  R-Squared          RMSE  Time Taken
Model                                                                            
KernelRidge                      1017.505539 -77.192734  56964.979303    0.076980
MLPRegressor                      948.574068 -71.890313  54999.615430    0.576711
LinearSVR                         932.119947 -70.624611  54520.004688    0.023275
GaussianProcessRegressor          523.525186 -39.194245  40841.970128    0.100775
QuantileRegressor                  17.161613  -0.243201   7182.827323    0.081644
SVR                                16.985645  -0.229665   7143.616910    0.039873
DummyRegressor                     14.230939  -0.017765   6499.027928    0.013008
NuSVR                              14.097255  -0.007481   6466.111890    0.029122
ElasticNetCV                        8.394396   0.431200   4858.524888    0.125515
LGBMRegressor                       7.683144   0.485912   4618.952152    0.13

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:     2518.9 | RMSE:     3261.9 | MAPE:  4.77%
FLAML Test (lgbm)              | MAE:     3456.4 | RMSE:     4513.3 | MAPE:  9.77%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:    10016.0 | RMSE:    12565.6 | MAPE: 27.99%
SF AutoETS                     | MAE:    10019.0 | RMSE:    12532.2 | MAPE: 27.91%
SF SeasonalNaive               | MAE:    10060.8 | RMSE:    12631.3 | MAPE: 28.18%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model          MAE         RMSE      MAPE
      FLAML (lgbm)  2518.891665  3261.884320  4.767742
 FLAML Test (lgbm)  3456.381835  4513.342309  9.774277
Seasonal Naive (7)  8287.500000 10911.447694 21.967102
Naive (Last Value)  9086.928571 11632.135564 23.328784
      SF AutoARIMA 10015.979492 12565.564373 27.992327
        SF AutoETS 10018.995117 12532.241300 27.911989
  SF SeasonalNaive 10060.786133 12631.295104 28.178504

Top 3:
             model         MAE         RMSE      MAPE
      FLAML (lgbm) 2518.891665  3261.884320  4.767742
 FLAML Test (lgbm) 3456.381835  4513.342309  9.774277
Seasonal Naive (7) 8287.500000 10911.447694 21.967102


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -1244.94, Std: 4338.25


## Interpretation and Business Insight

- Traffic volume has the most predictable weekly pattern of any domain
- Weekday commuter traffic is 30-40% higher than weekends
- Holidays cause dramatic drops as commuters stay home
- Summer has slightly higher volume (vacation travel offsets commuter loss)
- Weather events (not modeled) cause significant day-to-day variance

## Limitations

1. Synthetic — real traffic depends on weather, events, construction
2. No weather features (rain/snow dramatically reduces traffic)
3. Single location — different roads have different patterns
4. Daily granularity — peak hours matter more than daily totals
5. No special event data (sports, concerts)

## How to Improve This Project

1. Add weather data (rain, snow, visibility)
2. Use hourly data for peak-hour analysis
3. Add event calendar features
4. Model multiple locations with spatial correlation
5. Use real-time loop detector data for nowcasting

## Production Considerations

- Day-ahead and week-ahead traffic forecasting
- Integration with traffic management centers
- Construction zone impact prediction
- Public transit demand coordination

## Common Mistakes

1. Ignoring weather — rain reduces traffic 10-20%
2. Not modeling holidays separately (they break weekly patterns)
3. Using daily totals when hourly peaks matter
4. Treating all days of week as equal

## Mini Challenge / Exercises

1. Add a synthetic weather feature (rain indicator) and measure effect
2. Separate weekday and weekend models — compare accuracy
3. Detect anomalous traffic days (accidents, events)
4. Try predicting peak vs off-peak volume separately

## Final Summary / Key Takeaways

1. Traffic volume is highly predictable due to strong weekly patterns
2. Weekday/weekend distinction is the dominant signal
3. Holidays break normal patterns — need explicit handling
4. Weather (not included) is the biggest short-term disruptor
5. Hourly granularity matters more for operational use

In [18]:
import json
metrics = {
    'project': 'Traffic Volume Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Traffic Volume Forecasting — Complete ===")

Metrics saved.

=== Traffic Volume Forecasting — Complete ===
